# CNN ensemble trained on the pop-cosmos mock

The usual 1-parameter CNN ensemble (`znn.train_ensembles`, K-fold, MSE), trained and tested on the
pop-cosmos mock in its native COSMOS2020 bands, encoded on the LSST+Roman wavelength grid (see
`popcosmos_encoding.ipynb` for the encoding itself).

- **Bands:** the 23 COSMOS2020 bands inside the LSST ugrizy + Roman Y106/J129/H158 range (Ks and IRAC
  dropped); `lambda_range` fixed to that range, so the inputs share their bins with the challenge data.
- **Data:** disjoint random train and test samples of the mock with HSC i < `I_MAX`. The photometry is
  the mock's own noisy fluxes, at COSMOS2020 depth (much deeper than LSST in the optical).
- **p(z):** baseline (ensemble spread) and resampled (photometric errors propagated, assuming Gaussian
  noise on the fluxes with the quoted errors).
- **Output:** the ensemble is saved with `znn.save_ensemble_file` (members, bands, config with
  `lambda_range`, filter curves) to `MODEL_FILE`, and reloaded on re-runs. It is the pre-trained model for
  `CNN_photoz-ensemble-cardinal_popcosmos_pretraining.ipynb`.

In [ ]:
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from qp.metrics.pit import PIT

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

import znn  # run from the repo root so this picks up the local source

## Configuration

In [ ]:
MOCK_FILE = "mock_catalog_Ch1_26.h5"
FILTER_DIR = "data/popcosmos_filters"
MODEL_FILE = os.environ.get("CNNPZ_POPCOSMOS_MODEL", "models/popcosmos/popcosmos_ensemble_CNN_6layers.pkl")

I_MAX = 25.5  # magnitude cut in HSC i
N_TRAIN = 200_000  # training objects (split K ways inside train_ensembles)
N_TEST = 50_000  # held-out test objects, disjoint from the training sample
SEED = 42

N_SPLITS = 5  # ensemble members (K-fold); fine-tuning uses one fold per member
EPOCHS = 100  # max epochs per member (early stopping usually ends sooner)
N_SAMPLES = 50  # noise realizations S for the resampled prediction
NOISE_SEED = 42

# (band, filter file) in the mock's column order; see data/popcosmos_filters/SOURCES.md
MOCK_BANDS = [
    ("u", "u_megaprime_sagem.res"),
    ("g", "hsc_g.res"),
    ("r", "hsc_r.res"),
    ("i", "hsc_i.res"),
    ("z", "hsc_z.res"),
    ("y", "hsc_y.par"),
    ("Y", "uvista_y_cosmos.par"),
    ("J", "uvista_j_cosmos.par"),
    ("H", "uvista_h_cosmos.par"),
    ("Ks", "uvista_ks_cosmos.par"),
    *[
        (name, f"ia{name[2:]}_cosmos.par")
        for name in ["IB427", "IB464", "IA484", "IB505", "IA527", "IB574",
                     "IA624", "IA679", "IB709", "IA738", "IA767", "IB827"]
    ],
    ("NB711", "NB711.SuprimeCam.par"),
    ("NB816", "NB816.SuprimeCam.par"),
    ("ch1", "irac1_cosmos.par"),
    ("ch2", "irac2_cosmos.par"),
]
MOCK_COLUMN = {band: k for k, (band, _) in enumerate(MOCK_BANDS)}

# Wavelength span of the DC2LSST_{ugrizy} and roman_{Y106,J129,H158} curves used for the challenge
LAMBDA_RANGE = (3000.0, 18650.0)  # Å
REF_BAND = "i"
CONFIG = {**znn.DEFAULT_FEATURE_CONFIG, "lambda_range": LAMBDA_RANGE}


def response_range(curve, level=0.01):
    w, t = curve.T
    w = w[t > level * t.max()]
    return w.min(), w.max()


# keep the bands whose response (> 1% of peak) lies inside LAMBDA_RANGE
BANDS = {}
for band, file in MOCK_BANDS:
    lo, hi = response_range(np.loadtxt(f"{FILTER_DIR}/{file}"))
    if LAMBDA_RANGE[0] <= lo and hi <= LAMBDA_RANGE[1]:
        BANDS[band] = (f"mag_{band}", f"{FILTER_DIR}/{file}")
# magnitude column -> its error column, for the noise realizations (Gaussian in flux with the quoted error)
ERRORS = {column: f"{column}_err" for column, _ in BANDS.values()}
print(f"{len(BANDS)} bands:", list(BANDS))

## Load and encode the mock

Fluxes are in maggies; negative noisy fluxes become non-detections (`NaN`, treated as unobserved).
Redshift is column 15 of `sps_parameters`.

In [ ]:
with h5py.File(MOCK_FILE, "r") as f:
    fluxes = f["fluxes"][:]
    sigmas = f["flux_sigmas"][:]
    redshift = f["sps_parameters"][:, 15]

with np.errstate(divide="ignore", invalid="ignore"):
    detected = fluxes > 0
    mags = np.where(detected, -2.5 * np.log10(fluxes), np.nan)
    mag_errs = np.where(detected, 2.5 / np.log(10) * sigmas / fluxes, np.nan)

passing = np.flatnonzero(mags[:, MOCK_COLUMN[REF_BAND]] < I_MAX)
chosen = np.random.default_rng(SEED).permutation(passing)
idx = {"train": np.sort(chosen[:N_TRAIN]), "test": np.sort(chosen[N_TRAIN:N_TRAIN + N_TEST])}


def mock_catalog(rows):
    return pd.DataFrame(
        {
            **{column: mags[rows, MOCK_COLUMN[b]] for b, (column, _) in BANDS.items()},
            **{f"{column}_err": mag_errs[rows, MOCK_COLUMN[b]] for b, (column, _) in BANDS.items()},
            "redshift": redshift[rows],
        }
    )


train = mock_catalog(idx["train"])
test = mock_catalog(idx["test"])
print(f"{len(passing):,} mock galaxies with i < {I_MAX}; train {len(train):,}, test {len(test):,}")
del fluxes, sigmas, mags, mag_errs

filter_curves = znn.load_filter_curves(BANDS)
X_train, Y_train = znn.catalog_to_XY(train, BANDS, REF_BAND, CONFIG, filter_curves=filter_curves)
X_test, Y_test = znn.catalog_to_XY(test, BANDS, REF_BAND, CONFIG, filter_curves=filter_curves)
mag_i_test = test[f"mag_{REF_BAND}"].to_numpy()
print("train:", X_train.shape, "test:", X_test.shape)

## Train the ensemble (or load it if already trained)

In [ ]:
if os.path.exists(MODEL_FILE):
    trained_models, features = znn.load_ensemble_file(MODEL_FILE)
    assert features["config"] == CONFIG and list(features["bands"]) == list(BANDS), "MODEL_FILE was trained with other settings"
    histories = None
else:
    trained_models, histories = znn.train_ensembles(znn.build_model, X_train, Y_train, N_SPLITS=N_SPLITS, EPOCHS=EPOCHS)
    os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)
    znn.save_ensemble_file(MODEL_FILE, trained_models, BANDS, REF_BAND, CONFIG)

if histories is not None:
    znn.plot_ensemble_losses(histories, ylim=(0, 0.002))

## Test-set predictions

Two versions, as in `CNN_photoz-ensemble-cardinal_noise_realizations.ipynb`:
- **baseline**: the ensemble on the observed photometry (`znn.ensemble_predict`); the width is model spread only;
- **resampled, S = `N_SAMPLES`**: every member run on every one of S noise realizations of the test
  photometry (`znn.ensemble_predict_resampled`), so the width also carries the photometric errors.

The resampling assumes the noise on the fluxes is Gaussian with the quoted `flux_sigmas`. That is an
approximation for this mock: the actual noise is about 20% wider in its core than the quoted errors and
heavy-tailed (see `popcosmos_encoding.ipynb`), so the resampled widths should still come out somewhat
narrow, but less overconfident than the baseline.

In [ ]:
pz, components = {}, {}
pz["baseline"] = znn.ensemble_predict(trained_models, X_test)
name = f"resampled, S={N_SAMPLES}"
pz[name], components[name] = znn.ensemble_predict_resampled(
    trained_models,
    test,
    ERRORS,
    N_SAMPLES,
    np.random.default_rng(NOISE_SEED),
    BANDS,
    REF_BAND,
    config=CONFIG,
    filter_curves=filter_curves,
    return_components=True,
)

VERSIONS = list(pz)
z_point, width = {}, {}
for name in VERSIONS:
    z_point[name] = np.squeeze(pz[name].ancil["zmode"])
    width[name] = znn.predictions_mean_std(pz[name])[1].ravel()
    print(f"{name}: median p(z) width = {np.median(width[name]):.4f}")
RESAMPLED = VERSIONS[1]

## Point estimates

Δz = (z_phot − z_true) / (1 + z_true), with the p(z) mean as z_phot (for the resampled version, the mean
over the noise realizations).

In [ ]:
redshift_bins = np.linspace(0, 4, 11)
imag_bins = np.linspace(18, I_MAX, 11)
stats = {}
for name in VERSIONS:
    stats[name] = znn.get_all_stats(Y_test, z_point[name], mag_i_test, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins)
    znn.plot_stats(*stats[name], Y_test, z_point[name], redshift_bins, imag_bins, mag_i_test)
    plt.suptitle(name, y=1.02)
    plt.show()

display(Markdown(znn.stats_to_markdown(stats["baseline"][0], stats[RESAMPLED][0], data_title=("baseline", RESAMPLED))))

## Where does the width come from?

For the resampled version: median model and photometric parts of the width / (1 + z), in bins of i
magnitude and true redshift, against the actual scatter of Δz (biweight).

In [ ]:
def binned_median(x, values, bins):
    idx = np.digitize(x, bins) - 1
    return np.array([np.median(values[idx == i]) if np.any(idx == i) else np.nan for i in range(len(bins) - 1)])


fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
for ax, x, bins, binned, xlabel in [
    (axarr[0], mag_i_test, imag_bins, stats[RESAMPLED][2], "i magnitude"),
    (axarr[1], Y_test, redshift_bins, stats[RESAMPLED][1], "true redshift"),
]:
    centres = (bins[1:] + bins[:-1]) / 2
    for part, style in [("model", "-o"), ("photometric", "-s")]:
        ax.plot(centres, binned_median(x, components[RESAMPLED][part] / (1 + Y_test), bins), style, label=f"{part} part")
    ax.plot(centres, binned_median(x, width[RESAMPLED] / (1 + Y_test), bins), "-^", color="k", label="total width")
    ax.plot(centres, [s[2] for s in binned], "k--", label="actual scatter")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$\sigma / (1+z)$")
    ax.set_yscale("log")
    ax.legend(fontsize=10)
plt.suptitle(RESAMPLED, y=1.02)
plt.tight_layout()
plt.show()

## p(z) calibration: PIT

PIT = CDF of each object's p(z) at its true redshift; uniform for calibrated p(z). A U shape means p(z)
too narrow. `outlier` is the fraction of PIT values below 0.0001 or above 0.9999.

In [ ]:
pit_values, pit_stats = {}, {}
for name in VERSIONS:
    pit = PIT(pz[name], Y_test)
    pit_values[name] = pit.pit_samps
    pit_stats[name] = {
        "ks": pit.evaluate_PIT_KS().statistic,
        "CvM": pit.evaluate_PIT_CvM().statistic,
        "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
        "outlier": np.mean((pit.pit_samps < 0.0001) | (pit.pit_samps > 0.9999)),
    }

fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
q = np.linspace(0, 1, 101)
for name in VERSIONS:
    axarr[0].hist(pit_values[name], bins=50, range=(0, 1), density=True, histtype="step", lw=2, label=name)
    axarr[1].plot(q, np.quantile(pit_values[name], q), lw=2, label=name)
axarr[0].axhline(1, color="k", ls="--")
axarr[0].set_xlabel("PIT")
axarr[0].set_ylabel("density")
axarr[0].legend(fontsize=10)
axarr[1].plot([0, 1], [0, 1], "k--")
axarr[1].set_xlabel("uniform quantile")
axarr[1].set_ylabel("PIT quantile")
plt.tight_layout()
plt.show()

## Do the widths match the actual errors?

Median p(z) width / (1 + z) against the actual scatter of Δz, and the residuals divided by the predicted
width, which should follow a unit Gaussian.

In [ ]:
fig, axarr = plt.subplots(1, 3, figsize=[18, 5])
for k, name in enumerate(VERSIONS):
    rel_width = width[name] / (1 + Y_test)
    for ax, x, bins, binned in [
        (axarr[0], mag_i_test, imag_bins, stats[name][2]),
        (axarr[1], Y_test, redshift_bins, stats[name][1]),
    ]:
        centres = (bins[1:] + bins[:-1]) / 2
        ax.plot(centres, binned_median(x, rel_width, bins), "-o", color=f"C{k}", label=f"{name}: median width")
        ax.plot(centres, [s[2] for s in binned], "--", color=f"C{k}", label=f"{name}: actual scatter")
    axarr[2].hist(
        (z_point[name] - Y_test) / width[name], bins=np.linspace(-6, 6, 121), density=True, histtype="step", lw=2, label=name
    )
zz = np.linspace(-6, 6, 200)
axarr[2].plot(zz, np.exp(-0.5 * zz**2) / np.sqrt(2 * np.pi), "k--", label="unit Gaussian")
axarr[0].set_xlabel("i magnitude")
axarr[1].set_xlabel("true redshift")
for ax in axarr[:2]:
    ax.set_ylabel(r"$\sigma / (1+z)$")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
axarr[2].set_xlabel("(z_phot - z_true) / width")
axarr[2].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary = {}
for name in VERSIONS:
    bias, bias_err, scatter, outlier_rate, abs_outlier_rate = stats[name][0]
    summary[name] = {
        "bias": bias,
        "scatter": scatter,
        "outlier rate (3 sigma)": outlier_rate,
        "outlier rate (|dz| > 0.2)": abs_outlier_rate,
        "median p(z) width": np.median(width[name]),
        **{f"PIT {k}": v for k, v in pit_stats[name].items()},
    }
pd.DataFrame(summary).round(4)